<h1><strong>Customer Churn</strong></h1>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

# Suppressing warnings
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.regularizers import l2

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# General Style Settings
sns.set_style("whitegrid")  # Light grid background
sns.set_palette("viridis")  # Color palette
plt.rcParams["figure.figsize"] = (8, 4)  # Default figure size
plt.rcParams["axes.labelsize"] = 12  # Axis label size
plt.rcParams["axes.titlesize"] = 14  # Title size
plt.rcParams["xtick.labelsize"] = 11  # X-axis tick label size
plt.rcParams["ytick.labelsize"] = 11  # Y-axis tick label size
plt.rcParams["legend.fontsize"] = 11  # Legend font size
plt.rcParams["axes.edgecolor"] = "black"  # Border color of the plot
plt.rcParams["grid.alpha"] = 0.5  # Grid line transparency


def customize_plot(title="", xlabel="", ylabel=""):
    """
    Customizes the plot by adding a title, x-axis label, y-axis label,
    and setting grid and rotation properties.
    """
    plt.title(title, fontsize=14, fontweight="bold", color="darkblue")  # Set title
    plt.xlabel(xlabel, fontsize=12, fontweight="bold")  # Set x-axis label
    plt.ylabel(ylabel, fontsize=12, fontweight="bold")  # Set y-axis label
    plt.xticks(rotation=45)  # Rotate x-axis labels for better readability
    plt.grid(True, linestyle="--", alpha=0.6)  # Add dashed grid lines




In [ ]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [ ]:
df.head()

#### <span> Understanding the Data</span>


In [ ]:
df.sample(10)

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
df = df.dropna()
df.shape

In [ ]:
df.duplicated().sum()

#### <span>Data Manipulation</span>


In [ ]:
df = df.drop("customerID", axis=1)

In [ ]:
df[df["TotalCharges"] == ' ']

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [ ]:
df[df["TotalCharges"] == ' ']


In [ ]:
df.dtypes

In [ ]:
df['TotalCharges'] = df['TotalCharges'].replace([np.inf, -np.inf], np.nan)  
df = df.dropna(subset=['TotalCharges'])  


In [ ]:
df.isnull().sum()

In [ ]:
def outliers(df):

    for col in df.select_dtypes(include='number').columns:

        Q1 = df[col].quantile(0.25)

        Q3 = df[col].quantile(0.75)

        IQR = Q3 - Q1

        outliers = df[(df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))]



        print(f"{col} - Outliers:")

        print(f"Number of Outliers: {outliers.shape[0]}")

        print(outliers[[col]])

        print("---------------------------")

In [ ]:
outliers(df)

In [ ]:
df[df["tenure"] == 0]

In [ ]:
df.isnull().sum()


#### <span> Data Visualization</span>

In [ ]:
def plot_numeric_distributions(df):

    numeric_cols = df.select_dtypes(include=['number']).columns

    num_cols = len(numeric_cols)
    num_rows = (num_cols // 3) + 1

    plt.figure(figsize=(15, 5 * num_rows))

    for i, col in enumerate(numeric_cols, 1):
        plt.subplot(num_rows, 3, i)
        sns.histplot(df[col], kde=True, color='blue', bins=30)
        plt.title(f'Distribution of {col}', fontsize=12)
        plt.xlabel(col)
        plt.ylabel('Frequency')

    plt.tight_layout()
    plt.show()

In [ ]:
plot_numeric_distributions(df)

In [ ]:
def plot_categorical_barcharts(df, max_unique=30):
    categorical_columns = [
        col for col in df.select_dtypes(include=['object', 'category']).columns
        if df[col].nunique() < max_unique
    ]

    num_columns = len(categorical_columns)

    if num_columns == 0:
        print(f"No categorical columns with less than {max_unique} unique values.")
        return

    num_rows = math.ceil(num_columns / 3)
    num_cols = min(3, num_columns)

    fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, num_rows * 5))
    axes = axes.flatten()

    for i, col in enumerate(categorical_columns):
        df[col].value_counts().plot.bar(
            ax=axes[i],
            color='lightblue',
            edgecolor='black'
        )
        axes[i].set_title(f'Distribution of {col}')
        axes[i].set_ylabel('Count')
        axes[i].set_xlabel('Categories')

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

In [ ]:
plot_categorical_barcharts(df, max_unique=30)

In [ ]:
sns.countplot(x='gender', data=df)
plt.show()

In [ ]:
sns.countplot(x='PaymentMethod', hue='Churn', data=df)
plt.show()

In [ ]:
sns.countplot(x='Contract', hue='Churn', data=df)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
correlation_matrix = df.select_dtypes(include='number').corr()
sns.heatmap(data=correlation_matrix, annot=True, fmt=".2f", ax=ax)
ax.set_title("Correlation Matrix of Telco Customer Churn Dataset")
plt.show()


In [ ]:

sns.histplot(df[df['Churn'] == 'Yes']['tenure'], color="red", label="Churn", kde=True, binwidth=2)
sns.histplot(df[df['Churn'] == 'No']['tenure'], color="blue", label="Not Churn", kde=True, binwidth=2)

plt.legend()
plt.title("Tenure Distribution (Churn vs. Non-Churn Customers)")

plt.show()


In [ ]:

sns.histplot(df[df['Churn'] == 'Yes']['MonthlyCharges'], color="red", label="Churn", kde=True, bins=70)
sns.histplot(df[df['Churn'] == 'No']['MonthlyCharges'], color="blue", label="Not Churn", kde=True, bins=70)

plt.legend()
plt.title("Tenure Distribution (Total Charges vs. Non-Churn Customers)")

plt.show()


In [ ]:
sns.histplot(df[df['Churn'] == 'Yes']['SeniorCitizen'], color="red", label="Churn", kde=True, bins=70)
sns.histplot(df[df['Churn'] == 'No']['SeniorCitizen'], color="blue", label="Not Churn", kde=True, bins=70)

plt.legend()
plt.title("Tenure Distribution (Total Charges vs. Non-Churn Customers)")

plt.show()

In [ ]:
pd.crosstab(df['Partner'], df['Churn']).plot(kind='bar', stacked=True)
plt.title("StreamingTV VS Churn ")
plt.show()


In [ ]:
df.columns

#### <span> Data Preprocessing</span>

In [ ]:
df.Churn = df.Churn.map({'Yes': 1, 'No': 0})

In [ ]:
X = df[['Contract', 'InternetService', 'PaymentMethod', 'TotalCharges', 'tenure', 'OnlineSecurity', 'StreamingTV', 'PaperlessBilling', 'StreamingMovies', 'MultipleLines']]
y = df.Churn

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.20,stratify=y, random_state=42)

In [ ]:
df.columns

In [ ]:
# Define columns for different encoding strategies
binary_cols = ['PaperlessBilling']
ordinal_cols = ['Contract', 'InternetService']
nominal_cols = ['MultipleLines', 'OnlineSecurity', 'StreamingTV', 'StreamingMovies', 'PaymentMethod']
numeric_cols = ['TotalCharges', 'tenure']

# Define ordinal categories for ordered encoding
ordinal_mapping = [['Month-to-month', 'One year', 'Two year'],
                   ['No', 'DSL', 'Fiber optic']]

# Define transformers
binary_transformer = OneHotEncoder(drop='if_binary')  # Automatically encodes binary as 0/1
ordinal_transformer = OrdinalEncoder(categories=ordinal_mapping)
nominal_transformer = OneHotEncoder(drop='first')  # Drop first to avoid dummy variable trap
numeric_transformer = StandardScaler()  # Apply StandardScaler to numeric columns

# Apply transformations using ColumnTransformer
preprocessor = ColumnTransformer([
    ('binary', binary_transformer, binary_cols),
    ('ordinal', ordinal_transformer, ordinal_cols),
    ('nominal', nominal_transformer, nominal_cols),
    ('numeric', numeric_transformer, numeric_cols),
], remainder='passthrough')  # Keep any extra columns if needed

In [ ]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

In [ ]:
pd.DataFrame(X_train_transformed).head()

In [ ]:
pd.DataFrame(X_train_transformed).isnull().sum()

In [ ]:
def eval_metric(model, X_train, y_train, X_test, y_test):
    plt.figure(figsize=(8, 6))
    y_train_pred = np.array(np.where(model.predict(X_train)>0.5, 1, 0)).reshape((-1,))
    y_pred = np.array(np.where(model.predict(X_test)>0.5, 1, 0)).reshape((-1,))
    print("Train_Set")
    cfm = confusion_matrix(y_train, y_train_pred)
    sns.heatmap(cfm, annot=True, fmt='d', cmap='viridis')
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title("Train_Set Confusion Matrix")
    plt.show()
    print(classification_report(y_train, y_train_pred))
    eval_dict_train = classification_report(y_train, y_train_pred, output_dict=True)
    print()
    print("Test_Set")
    cfm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cfm, annot=True, fmt='d', cmap='viridis')
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title("Test_Set Confusion Matrix")
    plt.show()
    eval_dict_test = classification_report(y_test, y_pred, output_dict=True)
    print(classification_report(y_test, y_pred))
    print()
    return eval_dict_train, eval_dict_test

### <span> Modeling</span>

#### <span> Logistic model</span>

In [ ]:
logistic = LogisticRegression(class_weight='balanced', random_state=42)
logistic.fit(X_train_transformed , y_train)

In [ ]:
logistic_eval_train, logistic_eval_test = eval_metric(logistic, X_train_transformed, y_train, X_test_transformed, y_test)

#### Decision Tree

In [ ]:
tree = DecisionTreeClassifier( max_depth=8,random_state=42).fit(X_train_transformed, y_train)

In [ ]:
tree_eval_train, tree_eval_test = eval_metric(tree, X_train_transformed, y_train, X_test_transformed, y_test)

#### Random Forest

In [ ]:
forest = RandomForestClassifier( max_depth=10,random_state=42).fit(X_train_transformed, y_train)

In [ ]:
forest_eval_train, forest_eval_test  = eval_metric(forest, X_train_transformed, y_train, X_test_transformed, y_test)

#### <span> XGBoost</span>

In [ ]:
xgb = XGBClassifier(class_weight='balanced', n_estimators=100, max_depth=2).fit(X_train_transformed,y_train)

In [ ]:
xgb_eval_train, xgb_eval_test = eval_metric(xgb, X_train_transformed, y_train, X_test_transformed, y_test)


#### Artificial Neural Network

In [ ]:

nn_model = Sequential(
    [
        Dense(512, activation='relu'),
        Dense(256, activation='relu'),
        Dense(128, activation='relu'),
        Dense(1, activation='sigmoid')
    ]
)
nn_model.compile(optimizer='adam', loss=tf.keras.losses.BinaryCrossentropy(from_logits=False), metrics=[tf.keras.metrics.BinaryAccuracy()])

In [ ]:
X_train_tens = tf.convert_to_tensor(X_train_transformed)
y_train_tens = tf.convert_to_tensor(y_train)
nn_model.fit(X_train_tens, y_train_tens, epochs = 50, batch_size=200)

In [ ]:
nn_model_eval_train, nn_model_eval_test = eval_metric(nn_model, X_train_transformed, y_train, X_test_transformed, y_test)


### Comparing Models

In [ ]:
compare = pd.DataFrame({
    "Model": ["Logistic Regression","Decision Tree", "Random Forest", "XGBoost", "Neural Network"],
    "Accuracy": [logistic_eval_test['accuracy'], tree_eval_test['accuracy'], forest_eval_test['accuracy'], xgb_eval_test['accuracy'], nn_model_eval_test['accuracy']]
})


compare = compare.sort_values(by="Accuracy", ascending=False)


plt.figure(figsize=(12, 4))
ax = sns.barplot(x="Accuracy", y="Model", data=compare, palette="viridis")


for p in ax.patches:
    ax.annotate(f"{p.get_width():.3f}", 
                (p.get_x() + p.get_width(), p.get_y() + p.get_height()/2), 
                xytext=(5, 0), textcoords='offset points', ha="left", va="center", size=12, color='black')



plt.title("Comparison of Model Performance Based on Test Accuracy", size=16, weight='bold')
plt.xlabel('Test Accuracy', size=14)
plt.ylabel('Models', size=14)


plt.tight_layout()
plt.show()



### Final Model

In [ ]:
import pickle
final_model=forest

In [ ]:
from sklearn.metrics import roc_curve
fpr_rf, tpr_rf, thresholds = roc_curve(y_test, xgb.predict_proba(X_test_transformed)[:, 1])
plt.plot([0, 1], [0, 1], 'k--' )
plt.plot(fpr_rf, tpr_rf, label='Random Forest',color = "r")
plt.legend()
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Random Forest ROC Curve',fontsize=16)
plt.show()

In [ ]:
pickle.dump(final_model, open("model.pkl", "wb"))
pickle.dump(preprocessor, open("preprocessor.pkl", "wb"))